# 07 Error analysis and visualization

**Question.** Where do the models err, and is uncertainty informative?

**Type.** Exploratory analysis of validation predictions of one chosen run (set `EXPERIMENT_ID`). No test data.

In [ ]:
import json, sys
from pathlib import Path
REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
import os; os.chdir(REPO)
import numpy as np
import matplotlib.pyplot as plt
from ct_iqa.experiment.runs import latest_completed_run, read_predictions
EXPERIMENT_ID = "A_reference_resnet50"
run_dir = latest_completed_run(EXPERIMENT_ID)
print(run_dir if run_dir else "NO COMPLETED RUN FOUND (PENDING)")

In [ ]:
if run_dir:
    p = read_predictions(run_dir)
    y, f = np.array(p["label_0_4"]), np.array(p["prediction_0_4"]); r = f - y
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
    ax[0].scatter(y, f, s=8); ax[0].plot([0, 4], [0, 4], "k--"); ax[0].set_xlabel("label"); ax[0].set_ylabel("prediction")
    ax[1].hist(r, bins=20); ax[1].set_xlabel("residual (pred - label)")
    ax[2].scatter(y, np.abs(r), s=8); ax[2].set_xlabel("label"); ax[2].set_ylabel("|error|"); plt.tight_layout(); plt.show()
    worst = np.argsort(-np.abs(r))[:5]
    print([(p["filename"][i], round(y[i], 2), round(f[i], 2)) for i in worst])

In [ ]:
if run_dir and "log_var_normalized" in p:
    sigma = 4 * np.exp(0.5 * np.array(p["log_var_normalized"]))
    plt.scatter(sigma, np.abs(r), s=8); plt.xlabel("predicted sigma (0-4 scale)"); plt.ylabel("|error|"); plt.show()
    from ct_iqa.evaluation.intervals import uncertainty_error_correlation
    print("Spearman(sigma, |error|):", uncertainty_error_correlation(sigma, np.abs(r)))

Limitations to keep in mind: image-level split, 100 validation images, aleatoric-only uncertainty, labels are rater averages. See [LIMITATIONS_AND_THREATS](../docs/limitations-and-threats.md).